In [1]:
#데이터셋 로딩
import pandas as pd
import numpy as np
from sklearn import model_selection
from sklearn import metrics
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.tree import DecisionTreeClassifier

In [2]:
df = pd.read_csv("data2/train.csv")

print(df.shape)
df.head() #행과 열의수 확인

(26457, 20)


,index,gender,car,reality,child_num,income_total,income_type,edu_type,family_type,house_type,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,work_phone,phone,email,occyp_type,family_size,begin_month,credit
0,0,F,N,N,0,202500.0,Commercial associate,Higher education,Married,Municipal apartment,-13899,-4709,1,0,0,0,NaN,2.0,-6.0,1.0
1,1,F,N,Y,1,247500.0,Commercial associate,Secondary / secondary special,Civil marriage,House / apartment,-11380,-1540,1,0,0,1,Laborers,3.0,-5.0,1.0
2,2,M,Y,Y,0,450000.0,Working,Higher education,Married,House / apartment,-19087,-4434,1,0,1,0,Managers,2.0,-22.0,2.0
3,3,F,N,Y,0,202500.0,Commercial associate,Secondary / secondary special,Married,House / apartment,-15088,-2092,1,0,1,0,Sales staff,2.0,-37.0,0.0
4,4,F,Y,Y,0,157500.0,State servant,Higher education,Married,House / apartment,-15037,-2105,1,0,0,0,Managers,2.0,-26.0,2.0


In [3]:
import pandas as pd

# 직업군 카테고리 정의
occupation_categories = {
    '관리 및 지원': ['Managers', 'Core staff', 'HR staff', 'IT staff', 'Secretaries'],
    '기술 및 전문직': ['High skill tech staff', 'Accountants', 'Medicine staff', 'Realty agents'],
    '판매 및 서비스': ['Sales staff', 'Private service staff', 'Waiters/barmen staff', 'Cooking staff'],
    '노동직': ['Laborers', 'Low-skill Laborers', 'Cleaning staff'],
    '운전 및 보안': ['Drivers', 'Security staff']
}

# 카테고리를 역으로 매핑하여 직업명에 따른 분류 생성
occupation_encoding = {}
for category, jobs in occupation_categories.items():
    for job in jobs:
        occupation_encoding[job] = category

# `occyp_type`을 기준으로 `occyp_category`에 매핑된 카테고리 값 생성
df['occyp_category'] = df['occyp_type'].map(occupation_encoding)

# NaN 값을 -1로, 나머지는 카테고리별 인코딩 (관리 및 지원: 0, 기술 및 전문직: 1, ...)
df['occyp_category_encoded'] = df['occyp_category'].astype('category').cat.codes
df['occyp_category_encoded'] = df['occyp_category_encoded'].replace(-1, -1)  # NaN 값을 -1로 유지

# 결과 확인
df["occyp_category_encoded"].value_counts()



-1    8171
 2    5042
 0    5013
 4    3363
 1    2869
 3    1999
Name: occyp_category_encoded, dtype: int64

In [4]:
df = df.drop(['index', 'FLAG_MOBIL', 'email'], axis=1)
df

,gender,car,reality,child_num,income_total,income_type,edu_type,family_type,house_type,DAYS_BIRTH,DAYS_EMPLOYED,work_phone,phone,occyp_type,family_size,begin_month,credit,occyp_category,occyp_category_encoded
0,F,N,N,0,202500.0,Commercial associate,Higher education,Married,Municipal apartment,-13899,-4709,0,0,NaN,2.0,-6.0,1.0,NaN,-1
1,F,N,Y,1,247500.0,Commercial associate,Secondary / secondary special,Civil marriage,House / apartment,-11380,-1540,0,0,Laborers,3.0,-5.0,1.0,노동직,2
2,M,Y,Y,0,450000.0,Working,Higher education,Married,House / apartment,-19087,-4434,0,1,Managers,2.0,-22.0,2.0,관리 및 지원,0
3,F,N,Y,0,202500.0,Commercial associate,Secondary / secondary special,Married,House / apartment,-15088,-2092,0,1,Sales staff,2.0,-37.0,0.0,판매 및 서비스,4
4,F,Y,Y,0,157500.0,State servant,Higher education,Married,House / apartment,-15037,-2105,0,0,Managers,2.0,-26.0,2.0,관리 및 지원,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26452,F,N,N,2,225000.0,State servant,Secondary / secondary special,Married,House / apartment,-12079,-1984,0,0,Core staff,4.0,-2.0,1.0,관리 및 지원,0
26453,F,N,Y,1,180000.0,Working,Higher education,Separated,House / apartment,-15291,-2475,0,0,NaN,2.0,-47.0,2.0,NaN,-1
26454,F,Y,N,0,292500.0,Working,Secondary / secondary special,Civil marriage,With parents,-10082,-2015,0,0,Core staff,2.0,-25.0,2.0,관리 및 지원,0
26455,M,N,Y,0,171000.0,Working,Incomplete higher,Single / not married,House / apartment,-10145,-107,0,0,Laborers,1.0,-59.0,2.0,노동직,2


In [5]:
df["credit"].value_counts()

2.0    16968
1.0     6267
0.0     3222
Name: credit, dtype: int64

In [6]:
#결측치 확인
df.isnull().sum()

gender                       0
car                          0
reality                      0
child_num                    0
income_total                 0
income_type                  0
edu_type                     0
family_type                  0
house_type                   0
DAYS_BIRTH                   0
DAYS_EMPLOYED                0
work_phone                   0
phone                        0
occyp_type                8171
family_size                  0
begin_month                  0
credit                       0
occyp_category            8171
occyp_category_encoded       0
dtype: int64

In [7]:
#성별 gender
pd.crosstab(df['gender'], df['credit'], margins=True, margins_name='Total')

credit,0.0,1.0,2.0,Total
gender,,,,
F,2148,4220,11329,17697
M,1074,2047,5639,8760
Total,3222,6267,16968,26457


In [8]:
pd.crosstab(df['gender'], df['credit'], normalize='index')

credit,0.0,1.0,2.0
gender,,,
F,0.121377,0.238458,0.640165
M,0.122603,0.233676,0.643721


In [9]:
# 차 car
pd.crosstab(df['car'], df['credit'], margins=True, margins_name='Total')

credit,0.0,1.0,2.0,Total
car,,,,
N,1982,3990,10438,16410
Y,1240,2277,6530,10047
Total,3222,6267,16968,26457


In [10]:
pd.crosstab(df['car'], df['credit'], normalize='index')

credit,0.0,1.0,2.0
car,,,
N,0.12078,0.243144,0.636076
Y,0.12342,0.226635,0.649945


In [11]:
# 부동산 소유여부 reality
pd.crosstab(df['reality'], df['credit'], margins=True, margins_name='Total')

credit,0.0,1.0,2.0,Total
reality,,,,
N,1064,1935,5628,8627
Y,2158,4332,11340,17830
Total,3222,6267,16968,26457


In [12]:
pd.crosstab(df['reality'], df['credit'], normalize='index')

credit,0.0,1.0,2.0
reality,,,
N,0.123334,0.224296,0.652370
Y,0.121032,0.242961,0.636007


In [13]:
# 연간소득 income_total
pd.crosstab(df['income_total'], df['credit'], margins=True, margins_name='Total')

credit,0.0,1.0,2.0,Total
income_total,,,,
27000.0,0,1,1,2
29250.0,1,4,0,5
30150.0,0,0,2,2
31500.0,6,3,4,13
31531.5,0,0,1,1
...,...,...,...,...
990000.0,0,1,2,3
1125000.0,0,1,1,2
1350000.0,0,0,4,4


In [14]:
# 연간 소득 income_total
pd.crosstab(df['income_total'], df['credit'], normalize='index')

credit,0.0,1.0,2.0
income_total,,,
27000.0,0.000000,0.500000,0.500000
29250.0,0.200000,0.800000,0.000000
30150.0,0.000000,0.000000,1.000000
31500.0,0.461538,0.230769,0.307692
31531.5,0.000000,0.000000,1.000000
...,...,...,...
945000.0,0.250000,0.000000,0.750000
990000.0,0.000000,0.333333,0.666667
1125000.0,0.000000,0.500000,0.500000


In [15]:
# income_type
pd.crosstab(df['income_type'], df['credit'], margins=True, margins_name='Total')

credit,0.0,1.0,2.0,Total
income_type,,,,
Commercial associate,782,1344,4076,6202
Pensioner,536,1084,2829,4449
State servant,265,489,1400,2154
Student,0,3,4,7
Working,1639,3347,8659,13645
Total,3222,6267,16968,26457


In [16]:
# income_type
pd.crosstab(df['income_type'], df['credit'], normalize='index')

credit,0.0,1.0,2.0
income_type,,,
Commercial associate,0.126088,0.216704,0.657207
Pensioner,0.120477,0.243650,0.635873
State servant,0.123027,0.227019,0.649954
Student,0.000000,0.428571,0.571429
Working,0.120117,0.245291,0.634591


In [17]:
# child_num
pd.crosstab(df['child_num'], df['credit'], margins=True, margins_name='Total')

credit,0.0,1.0,2.0,Total
child_num,,,,
0,2233,4322,11785,18340
1,682,1313,3391,5386
2,268,535,1559,2362
3,29,79,198,306
4,8,12,27,47
5,2,6,2,10
7,0,0,2,2
14,0,0,3,3
19,0,0,1,1


In [18]:
pd.crosstab(df['child_num'], df['credit'], normalize='index')

credit,0.0,1.0,2.0
child_num,,,
0,0.121756,0.235660,0.642585
1,0.126625,0.243780,0.629595
2,0.113463,0.226503,0.660034
3,0.094771,0.258170,0.647059
4,0.170213,0.255319,0.574468
5,0.200000,0.600000,0.200000
7,0.000000,0.000000,1.000000
14,0.000000,0.000000,1.000000
19,0.000000,0.000000,1.000000


In [19]:
# edu_type
pd.crosstab(df['edu_type'], df['credit'], margins=True, margins_name='Total')

credit,0.0,1.0,2.0,Total
edu_type,,,,
Academic degree,2,7,14,23
Higher education,909,1751,4502,7162
Incomplete higher,114,246,660,1020
Lower secondary,28,59,170,257
Secondary / secondary special,2169,4204,11622,17995
Total,3222,6267,16968,26457


In [20]:
pd.crosstab(df['edu_type'], df['credit'], normalize='index')

credit,0.0,1.0,2.0
edu_type,,,
Academic degree,0.086957,0.304348,0.608696
Higher education,0.126920,0.244485,0.628595
Incomplete higher,0.111765,0.241176,0.647059
Lower secondary,0.108949,0.229572,0.661479
Secondary / secondary special,0.120533,0.233620,0.645846


In [21]:
# family_type	
pd.crosstab(df['family_type'], df['credit'], margins=True, margins_name='Total')

credit,0.0,1.0,2.0,Total
family_type,,,,
Civil marriage,288,539,1296,2123
Married,2213,4140,11843,18196
Separated,193,349,997,1539
Single / not married,402,940,2154,3496
Widow,126,299,678,1103
Total,3222,6267,16968,26457


In [22]:
pd.crosstab(df['family_type'], df['credit'], normalize='index')

credit,0.0,1.0,2.0
family_type,,,
Civil marriage,0.135657,0.253886,0.610457
Married,0.121620,0.227523,0.650857
Separated,0.125406,0.226771,0.647823
Single / not married,0.114989,0.268879,0.616133
Widow,0.114234,0.271079,0.614687


In [23]:
# house_type
pd.crosstab(df['house_type'], df['credit'], margins=True, margins_name='Total')

credit,0.0,1.0,2.0,Total
house_type,,,,
Co-op apartment,14,30,66,110
House / apartment,2873,5569,15211,23653
Municipal apartment,110,160,548,818
Office apartment,24,48,118,190
Rented apartment,50,147,232,429
With parents,151,313,793,1257
Total,3222,6267,16968,26457


In [24]:
pd.crosstab(df['house_type'], df['credit'], normalize='index')

credit,0.0,1.0,2.0
house_type,,,
Co-op apartment,0.127273,0.272727,0.600000
House / apartment,0.121465,0.235446,0.643090
Municipal apartment,0.134474,0.195599,0.669927
Office apartment,0.126316,0.252632,0.621053
Rented apartment,0.116550,0.342657,0.540793
With parents,0.120127,0.249006,0.630867


In [25]:
# DAYS_BIRTH
pd.crosstab(df['DAYS_BIRTH'], df['credit'], margins=True, margins_name='Total')

credit,0.0,1.0,2.0,Total
DAYS_BIRTH,,,,
-25152,0,0,1,1
-25140,3,0,0,3
-25099,0,0,1,1
-25088,1,0,0,1
-24970,0,2,0,2
...,...,...,...,...
-7959,0,0,1,1
-7757,1,2,0,3
-7723,0,2,0,2


In [26]:
pd.crosstab(df['DAYS_BIRTH'], df['credit'], normalize='index')

credit,0.0,1.0,2.0
DAYS_BIRTH,,,
-25152,0.000000,0.000000,1.0
-25140,1.000000,0.000000,0.0
-25099,0.000000,0.000000,1.0
-25088,1.000000,0.000000,0.0
-24970,0.000000,1.000000,0.0
...,...,...,...
-8041,0.000000,0.000000,1.0
-7959,0.000000,0.000000,1.0
-7757,0.333333,0.666667,0.0


In [27]:
# DAYS_EMPLOYED
pd.crosstab(df['DAYS_EMPLOYED'], df['credit'], margins=True, margins_name='Total')

credit,0.0,1.0,2.0,Total
DAYS_EMPLOYED,,,,
-15713,0,0,1,1
-15661,0,0,2,2
-15072,1,0,2,3
-15038,5,3,6,14
-14887,3,1,1,5
...,...,...,...,...
-65,0,0,1,1
-43,0,1,0,1
-17,0,0,2,2


In [28]:
pd.crosstab(df['DAYS_EMPLOYED'], df['credit'], normalize='index')

credit,0.0,1.0,2.0
DAYS_EMPLOYED,,,
-15713,0.000000,0.000000,1.000000
-15661,0.000000,0.000000,1.000000
-15072,0.333333,0.000000,0.666667
-15038,0.357143,0.214286,0.428571
-14887,0.600000,0.200000,0.200000
...,...,...,...
-70,0.000000,0.000000,1.000000
-65,0.000000,0.000000,1.000000
-43,0.000000,1.000000,0.000000


In [29]:
# work_phone
pd.crosstab(df['work_phone'], df['credit'], margins=True, margins_name='Total')

credit,0.0,1.0,2.0,Total
work_phone,,,,
0,2493,4844,13174,20511
1,729,1423,3794,5946
Total,3222,6267,16968,26457


In [30]:
pd.crosstab(df['work_phone'], df['credit'], normalize='index')

credit,0.0,1.0,2.0
work_phone,,,
0,0.121545,0.236166,0.642290
1,0.122603,0.239321,0.638076


In [31]:
# phone
pd.crosstab(df['phone'], df['credit'], margins=True, margins_name='Total')
pd.crosstab(df['phone'], df['credit'], normalize='index')

credit,0.0,1.0,2.0
phone,,,
0,0.120234,0.241538,0.638228
1,0.125498,0.225690,0.648812


In [32]:
# occyp_type
pd.crosstab(df['occyp_type'], df['credit'], margins=True, margins_name='Total')

credit,0.0,1.0,2.0,Total
occyp_type,,,,
Accountants,118,227,557,902
Cleaning staff,40,93,270,403
Cooking staff,58,110,289,457
Core staff,347,622,1677,2646
Drivers,187,358,1030,1575
HR staff,7,4,51,62
High skill tech staff,123,270,647,1040
IT staff,8,10,23,41
Laborers,586,1082,2844,4512


In [33]:
pd.crosstab(df['occyp_type'], df['credit'], normalize='index')

credit,0.0,1.0,2.0
occyp_type,,,
Accountants,0.130820,0.251663,0.617517
Cleaning staff,0.099256,0.230769,0.669975
Cooking staff,0.126915,0.240700,0.632385
Core staff,0.131141,0.235072,0.633787
Drivers,0.118730,0.227302,0.653968
HR staff,0.112903,0.064516,0.822581
High skill tech staff,0.118269,0.259615,0.622115
IT staff,0.195122,0.243902,0.560976
Laborers,0.129876,0.239805,0.630319


In [34]:
df['occyp_type'].value_counts()

Laborers                 4512
Core staff               2646
Sales staff              2539
Managers                 2167
Drivers                  1575
High skill tech staff    1040
Accountants               902
Medicine staff            864
Cooking staff             457
Security staff            424
Cleaning staff            403
Private service staff     243
Low-skill Laborers        127
Waiters/barmen staff      124
Secretaries                97
Realty agents              63
HR staff                   62
IT staff                   41
Name: occyp_type, dtype: int64

In [35]:
# family_size
pd.crosstab(df['family_size'], df['credit'], margins=True, margins_name='Total')
pd.crosstab(df['family_size'], df['credit'], normalize='index')

credit,0.0,1.0,2.0
family_size,,,
1.0,0.118418,0.257976,0.623605
2.0,0.121934,0.230328,0.647739
3.0,0.131693,0.234888,0.633420
4.0,0.111062,0.230088,0.658850
5.0,0.092784,0.254296,0.652921
6.0,0.159091,0.272727,0.568182
7.0,0.222222,0.666667,0.111111
9.0,0.000000,0.000000,1.000000
15.0,0.000000,0.000000,1.000000


In [36]:
# 신용카드 발급 월 begin_month
pd.crosstab(df['begin_month'], df['credit'], margins=True, margins_name='Total')

credit,0.0,1.0,2.0,Total
begin_month,,,,
-60.0,22,41,172,235
-59.0,21,49,172,242
-58.0,25,47,172,244
-57.0,22,37,169,228
-56.0,34,36,184,254
...,...,...,...,...
-3.0,109,470,14,593
-2.0,87,390,1,478
-1.0,96,319,0,415


In [37]:
pd.crosstab(df['begin_month'], df['credit'], normalize='index')

credit,0.0,1.0,2.0
begin_month,,,
-60.0,0.093617,0.174468,0.731915
-59.0,0.086777,0.202479,0.710744
-58.0,0.102459,0.192623,0.704918
-57.0,0.096491,0.162281,0.741228
-56.0,0.133858,0.141732,0.724409
...,...,...,...
-4.0,0.134650,0.423698,0.441652
-3.0,0.183811,0.792580,0.023609
-2.0,0.182008,0.815900,0.002092


In [38]:
import pandas as pd

table = pd.crosstab(
    [df['occyp_type'], df['income_total']],
    df['credit'],
    normalize='index'
)

print(table)


credit                                  0.0       1.0       2.0
occyp_type           income_total                              
Accountants          45000.0       0.333333  0.333333  0.333333
                     67500.0       0.370370  0.444444  0.185185
                     72000.0       0.000000  0.500000  0.500000
                     76500.0       0.000000  0.800000  0.200000
                     85500.0       0.000000  0.250000  0.750000
...                                     ...       ...       ...
Waiters/barmen staff 202500.0      0.000000  0.250000  0.750000
                     225000.0      0.166667  0.333333  0.500000
                     270000.0      0.000000  0.125000  0.875000
                     292500.0      0.000000  0.000000  1.000000
                     360000.0      0.000000  0.400000  0.600000

[810 rows x 3 columns]
